# 🚢 Titanic Survival Analysis
**Author:** Elnissi Williams  
**Tools:** Python · Pandas · NumPy · Seaborn · Matplotlib  

## Overview
This project explores the Titanic passenger dataset to uncover the factors that influenced survival during the disaster. Through data cleaning, exploratory data analysis (EDA), and visualisation, we investigate how passenger class, gender, age, and fare affected the likelihood of survival.

## Dataset
The dataset contains **891 passenger records** with 15 features including demographics, ticket information, and survival outcome.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print("Libraries loaded successfully.")


## 1. Load & Inspect the Data

In [ ]:
df = sns.load_dataset('titanic')

print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumn names:\n{list(df.columns)}")
df.head()


In [ ]:
print("Data Types:")
print(df.dtypes)
print(f"\nBasic Statistics:")
df.describe()


## 2. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print("Columns with missing values:")
print(missing_df)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False,
            cmap='viridis', ax=ax)
ax.set_title('Missing Value Heatmap — Yellow = Missing', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Features')
plt.tight_layout()
plt.show()


## 3. Data Cleaning

**Strategy:**
- `age` — fill missing values with the **median age** (more robust to outliers than mean)
- `embarked` — fill 2 missing entries with the **mode** (most frequent port)
- `deck` — drop this column (77% missing — too sparse to be useful)


In [ ]:
# Fill age with median
df['age'] = df['age'].fillna(df['age'].median())

# Fill embarked with mode
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode()[0])

# Drop deck (77% missing)
df.drop(columns=['deck'], inplace=True)

# Verify
print(f"Remaining missing values: {df.isnull().sum().sum()}")
print(f"\nDataset shape after cleaning: {df.shape}")


## 4. Survival Overview

In [ ]:
survived_counts = df['survived'].value_counts()
survival_rate = df['survived'].mean() * 100

print(f"Total passengers: {len(df)}")
print(f"Survived: {survived_counts[1]} ({survival_rate:.1f}%)")
print(f"Did not survive: {survived_counts[0]} ({100 - survival_rate:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
axes[0].pie(survived_counts,
            labels=['Did Not Survive', 'Survived'],
            autopct='%1.1f%%',
            colors=['#E94F37', '#44BBA4'],
            startangle=90,
            explode=[0, 0.07],
            shadow=True)
axes[0].set_title('Overall Survival Rate', fontweight='bold', fontsize=13)

# Bar chart
bars = axes[1].bar(['Did Not Survive', 'Survived'],
                   survived_counts.values,
                   color=['#E94F37', '#44BBA4'],
                   edgecolor='white', width=0.5)
axes[1].set_title('Survival Count', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Number of Passengers')
for bar, val in zip(bars, survived_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 5, str(val),
                 ha='center', fontsize=12, fontweight='bold')

plt.suptitle('Titanic Survival Overview', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Survival by Gender

In [ ]:
gender_survival = df.groupby('sex')['survived'].agg(['sum', 'count', 'mean'])
gender_survival.columns = ['Survived', 'Total', 'Survival Rate']
gender_survival['Survival Rate'] = (gender_survival['Survival Rate'] * 100).round(1)
print(gender_survival)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Count plot
sns.countplot(x='survived', hue='sex', data=df, ax=axes[0],
              palette={'male': '#2E86AB', 'female': '#F18F01'})
axes[0].set_title('Survival Count by Gender', fontweight='bold')
axes[0].set_xlabel('Survived (0 = No, 1 = Yes)')
axes[0].set_ylabel('Count')
axes[0].legend(title='Gender')

# Survival rate bar chart
gender_survival['Survival Rate'].plot(kind='bar', ax=axes[1],
                                       color=['#2E86AB', '#F18F01'],
                                       edgecolor='white', width=0.5)
axes[1].set_title('Survival Rate by Gender (%)', fontweight='bold')
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_xlabel('Gender')
axes[1].tick_params(axis='x', rotation=0)
for i, val in enumerate(gender_survival['Survival Rate']):
    axes[1].text(i, val + 0.5, f'{val}%', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Gender and Survival', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey insight: Female passengers had a significantly higher survival rate than male passengers.")


## 6. Survival by Passenger Class

In [ ]:
class_survival = df.groupby('pclass')['survived'].agg(['sum', 'count', 'mean'])
class_survival.columns = ['Survived', 'Total', 'Survival Rate']
class_survival['Survival Rate'] = (class_survival['Survival Rate'] * 100).round(1)
class_survival.index = ['1st Class', '2nd Class', '3rd Class']
print(class_survival)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.countplot(x='pclass', hue='survived', data=df, ax=axes[0],
              palette={0: '#E94F37', 1: '#44BBA4'})
axes[0].set_title('Survival Count by Passenger Class', fontweight='bold')
axes[0].set_xlabel('Passenger Class')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['1st Class', '2nd Class', '3rd Class'])
axes[0].legend(title='Survived', labels=['No', 'Yes'])

bars = axes[1].bar(class_survival.index,
                   class_survival['Survival Rate'],
                   color=['#44BBA4', '#F18F01', '#E94F37'],
                   edgecolor='white', width=0.5)
axes[1].set_title('Survival Rate by Passenger Class (%)', fontweight='bold')
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_xlabel('Passenger Class')
for bar, val in zip(bars, class_survival['Survival Rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5, f'{val}%',
                 ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Passenger Class and Survival', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey insight: 1st class passengers had the highest survival rate — nearly double that of 3rd class.")


## 7. Age Distribution and Survival

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution by survival
for label, color in zip([0, 1], ['#E94F37', '#44BBA4']):
    subset = df[df['survived'] == label]['age']
    axes[0].hist(subset, bins=25, alpha=0.6,
                 label='Did Not Survive' if label == 0 else 'Survived',
                 color=color, edgecolor='white')
axes[0].set_title('Age Distribution by Survival Outcome', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].legend()

# KDE plot
for label, color in zip([0, 1], ['#E94F37', '#44BBA4']):
    subset = df[df['survived'] == label]['age']
    axes[1].plot(subset.sort_values(),
                 np.linspace(0, 1, len(subset)),
                 label='Did Not Survive' if label == 0 else 'Survived',
                 color=color, linewidth=2)
axes[1].set_title('Cumulative Age Distribution by Survival', fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Cumulative Proportion')
axes[1].legend()

plt.suptitle('Age and Survival', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Median age of survivors: {df[df['survived']==1]['age'].median():.1f} years")
print(f"Median age of non-survivors: {df[df['survived']==0]['age'].median():.1f} years")


In [ ]:
# Create age groups
bins = [0, 12, 18, 35, 60, 100]
labels = ['Child (0-12)', 'Teen (13-18)', 'Young Adult (19-35)', 'Adult (36-60)', 'Senior (60+)']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)

age_survival = df.groupby('age_group', observed=True)['survived'].agg(['mean', 'count'])
age_survival.columns = ['Survival Rate', 'Total']
age_survival['Survival Rate'] = (age_survival['Survival Rate'] * 100).round(1)
print(age_survival)

bars = plt.bar(age_survival.index,
               age_survival['Survival Rate'],
               color=['#44BBA4', '#2E86AB', '#F18F01', '#A23B72', '#E94F37'],
               edgecolor='white', width=0.6)
plt.title('Survival Rate by Age Group', fontweight='bold', fontsize=13)
plt.ylabel('Survival Rate (%)')
plt.xlabel('Age Group')
plt.xticks(rotation=15)
for bar, val in zip(bars, age_survival['Survival Rate']):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.4, f'{val}%',
             ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey insight: Children had the highest survival rate, consistent with 'women and children first' evacuation priority.")


## 8. Fare and Survival

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
df.boxplot(column='fare', by='survived', ax=axes[0],
           patch_artist=True)
axes[0].set_title('Fare Distribution by Survival', fontweight='bold')
axes[0].set_xlabel('Survived (0 = No, 1 = Yes)')
axes[0].set_ylabel('Fare (£)')
plt.sca(axes[0])
plt.title('Fare Distribution by Survival')

# Average fare
avg_fare = df.groupby('survived')['fare'].mean()
bars = axes[1].bar(['Did Not Survive', 'Survived'],
                   avg_fare.values,
                   color=['#E94F37', '#44BBA4'],
                   edgecolor='white', width=0.5)
axes[1].set_title('Average Fare by Survival Outcome', fontweight='bold')
axes[1].set_ylabel('Average Fare (£)')
for bar, val in zip(bars, avg_fare.values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5, f'£{val:.2f}',
                 ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Ticket Fare and Survival', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Average fare (survived): £{df[df['survived']==1]['fare'].mean():.2f}")
print(f"Average fare (did not survive): £{df[df['survived']==0]['fare'].mean():.2f}")
print("\nKey insight: Survivors paid on average significantly more — reflecting the correlation between passenger class and survival.")


## 9. Embarkation Port and Survival

In [ ]:
embark_survival = df.groupby('embark_town')['survived'].agg(['mean', 'count'])
embark_survival.columns = ['Survival Rate', 'Total']
embark_survival['Survival Rate'] = (embark_survival['Survival Rate'] * 100).round(1)
print(embark_survival)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.countplot(x='embark_town', hue='survived', data=df, ax=axes[0],
              palette={0: '#E94F37', 1: '#44BBA4'},
              order=['Southampton', 'Cherbourg', 'Queenstown'])
axes[0].set_title('Passenger Count by Embarkation Port', fontweight='bold')
axes[0].set_xlabel('Port')
axes[0].set_ylabel('Count')
axes[0].legend(title='Survived', labels=['No', 'Yes'])

bars = axes[1].bar(embark_survival.index,
                   embark_survival['Survival Rate'],
                   color=['#2E86AB', '#F18F01', '#A23B72'],
                   edgecolor='white', width=0.5)
axes[1].set_title('Survival Rate by Embarkation Port (%)', fontweight='bold')
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_xlabel('Port')
for bar, val in zip(bars, embark_survival['Survival Rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5, f'{val}%',
                 ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Embarkation Port and Survival', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 10. Correlation Heatmap

In [ ]:
numeric_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Correlation Matrix — Key Variables', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print("\nKey insight: Passenger class (pclass) has the strongest negative correlation with survival — higher class (lower number) = higher survival rate.")


## 11. Class × Gender Combined Analysis

In [ ]:
pivot = df.pivot_table(values='survived', index='pclass', columns='sex', aggfunc='mean') * 100
pivot.index = ['1st Class', '2nd Class', '3rd Class']
pivot = pivot.round(1)
print("Survival Rate (%) by Class and Gender:")
print(pivot)

fig, ax = plt.subplots(figsize=(9, 6))
x = np.arange(3)
w = 0.35
bars1 = ax.bar(x - w/2, pivot['female'], w,
               label='Female', color='#F18F01', edgecolor='white', alpha=0.9)
bars2 = ax.bar(x + w/2, pivot['male'], w,
               label='Male', color='#2E86AB', edgecolor='white', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(['1st Class', '2nd Class', '3rd Class'])
ax.set_title('Survival Rate (%) by Passenger Class and Gender', fontweight='bold', fontsize=13)
ax.set_ylabel('Survival Rate (%)')
ax.legend(title='Gender')
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5, f'{bar.get_height():.0f}%',
            ha='center', fontsize=9)
plt.tight_layout()
plt.show()


## 12. Key Findings & Summary

| Factor | Finding |
|---|---|
| **Overall survival rate** | 38.4% of passengers survived |
| **Gender** | Female survival rate (~74%) was far higher than male (~19%) |
| **Passenger class** | 1st class: ~63% survival · 2nd class: ~47% · 3rd class: ~24% |
| **Age** | Children (0–12) had the highest survival rate among age groups |
| **Fare** | Survivors paid on average ~2× more than non-survivors |
| **Strongest correlation** | Passenger class is the single strongest predictor of survival |
| **Class × Gender** | 1st class females had the highest survival (~97%) · 3rd class males had the lowest (~16%) |

### Conclusion
The Titanic disaster was not random — survival was strongly shaped by socioeconomic status, gender, and age. The "women and children first" protocol is clearly reflected in the data, as is the advantage held by first-class passengers who had better access to lifeboats and were located closer to the upper decks. These findings demonstrate how exploratory data analysis can surface meaningful patterns from historical data using Python and visualisation tools.
